In [ ]:
import os
import getpass
from github import Github
from git import Repo
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langsmith import traceable

In [ ]:
if "LANGSMITH_API_KEY" not in os.environ:
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LangSmith API key: ")

if "GITHUB_TOKEN" not in os.environ:
    os.environ["GITHUB_TOKEN"] = getpass.getpass("Enter GitHub Token: ")

GITHUB_REPO = "username/repo"
LOCAL_PATH = "./autocoder_repo"

gh = Github(os.environ["GITHUB_TOKEN"])
repo = gh.get_repo(GITHUB_REPO)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

prompt = PromptTemplate(
    input_variables=["description", "codebase"],
    template="""
You are AutoCoder. Based on the user's request, modify the given codebase.

Request:
{description}

Existing Codebase:
{codebase}

Return only the updated file content (Python).
"""
)

@traceable
def generate_code(description, codebase):
    chain = prompt | llm
    return chain.invoke({"description": description, "codebase": codebase}).content

def autocoder(description):
    if not os.path.exists(LOCAL_PATH):
        Repo.clone_from(repo.clone_url, LOCAL_PATH)
    local_repo = Repo(LOCAL_PATH)

    file_path = os.path.join(LOCAL_PATH, "main.py")
    with open(file_path, "r") as f:
        codebase = f.read()

    updated_code = generate_code(description, codebase)

    with open(file_path, "w") as f:
        f.write(updated_code)

    branch_name = f"autocoder-{description[:10].replace(' ', '-')}"
    local_repo.git.checkout("HEAD", b=branch_name)
    local_repo.git.add(A=True)
    local_repo.git.commit(m=f"AutoCoder Update: {description}")
    origin = local_repo.remote(name="origin")
    origin.push(refspec=f"{branch_name}:{branch_name}")

    pr = repo.create_pull(
        title=f"AutoCoder PR: {description}",
        body="This PR was auto-generated by AutoCoder 🤖",
        head=branch_name,
        base="main"
    )
    print(f"✅ Pull Request created: {pr.html_url}")

if __name__ == "__main__":
    task = input("Describe the change you want (e.g., 'Add sleep inducer function'): ")
    autocoder(task)